In [0]:
import time
import requests
from datetime import datetime, timezone

In [0]:
UNIFIED_PIPELINE_ID = "4324dd07-23c0-43fe-9311-5be56aac3299"

BRONZE_TABLE = "dbr_dev.skybound_bronze.flights_bronze"
IDLE_TIMEOUT_SEC = 120  
IDLE_CHECK_SEC = 10      

In [0]:
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = ctx.apiUrl().get()
token = ctx.apiToken().get()
headers = {"Authorization": f"Bearer {token}"}

BUSY_STATES = {"RUNNING", "STARTING", "DEPLOYING", "SETTING_UP", "WAITING_FOR_RESOURCES", "INITIALIZING"}

def get_max_timestamp():
    row = spark.sql(f"SELECT max(ingestion_timestamp) AS max_ts FROM {BRONZE_TABLE}").collect()[0]
    return str(row.max_ts)

def start_pipeline(pipeline_id):
    resp = requests.post(
        f"{host}/api/2.0/pipelines/{pipeline_id}/updates",
        headers=headers, json={"full_refresh": False}
    )
    return resp.status_code == 200

def stop_pipeline(pipeline_id):
    requests.post(f"{host}/api/2.0/pipelines/{pipeline_id}/stop", headers=headers)

def get_pipeline_state(pipeline_id):
    resp = requests.get(f"{host}/api/2.0/pipelines/{pipeline_id}", headers=headers)
    return resp.json().get("state", "UNKNOWN") if resp.status_code == 200 else "UNKNOWN"

def wait_for_completion(pipeline_id):
    """Block until the pipeline finishes the current update."""
    while get_pipeline_state(pipeline_id) in BUSY_STATES:
        time.sleep(5)

print("=" * 60)
print("Starting pipeline (first run may take 3-5 min for cluster)...")
print("=" * 60)

start_pipeline(UNIFIED_PIPELINE_ID)
print(f"[{datetime.now().strftime('%H:%M:%S')}] Pipeline triggered. Waiting for first batch...")
wait_for_completion(UNIFIED_PIPELINE_ID)
print(f"[{datetime.now().strftime('%H:%M:%S')}] First batch done.")

prev_max_ts = get_max_timestamp()
last_data_time = time.time()
print(f"Current max_ts: {prev_max_ts[:19]}")
print("-" * 60)

try:
    while True:
        current_max_ts = get_max_timestamp()
        now = time.time()

        if current_max_ts != prev_max_ts:
            prev_max_ts = current_max_ts
            last_data_time = now
            print(f"[{datetime.now().strftime('%H:%M:%S')}] New data → triggering batch (max_ts: {current_max_ts[:19]})")
            start_pipeline(UNIFIED_PIPELINE_ID)
            wait_for_completion(UNIFIED_PIPELINE_ID)
            print(f"[{datetime.now().strftime('%H:%M:%S')}] Batch complete.")
        else:
            idle_sec = int(now - last_data_time)
            print(f"[{datetime.now().strftime('%H:%M:%S')}] No new data — idle {idle_sec}s / {IDLE_TIMEOUT_SEC}s")

            if idle_sec >= IDLE_TIMEOUT_SEC:
                print(f"\n{'=' * 60}")
                print(f"Idle timeout reached. Stopping pipeline...")
                stop_pipeline(UNIFIED_PIPELINE_ID)
                print(f"Done.")
                print(f"{'=' * 60}")
                break

            time.sleep(IDLE_CHECK_SEC)

except KeyboardInterrupt:
    print(f"\nManual stop. Stopping pipeline...")
    stop_pipeline(UNIFIED_PIPELINE_ID)
    print("Pipeline stopped.")